# **SCAPES**

Follow this notebook to train your own SCAPES model and run it locally or online using the GUI. This notebook will not give you a deep understanding of how SCAPES works. If you are interested in getting more technical details, refer to the notebook `tutorial_full.ipynb` instead.

This single notebook covers all three SCAPES tasks end-to-end:
1. **Dataprep** — extract audio atoms and precompute annotations
2. **Training** — train a FlowModel + LocalEncoder
3. **Inference** — reconstruct, semantically reconstruct, and interpolate audio

This notebook works in Colab out of the box. If running locally, we strongly recommend copying it into a clean directory, creating a virtual environment, and running it there. The notebook will install all dependencies, including the SCAPES code, automatically.

# **0. Preamble**

## Check if your runtime has a GPU

SCAPES uses a transformer architecture that benefits significantly from GPU acceleration. Check your GPU status using the following cell.

In [ ]:
!nvidia-smi

## Install the model

Run this cell to install SCAPES and its dependencies. After installation, you may be prompted to restart the kernel — if so, restart and run this cell again.

In [ ]:
# Clone the repo
!git clone https://github.com/cordutie/SCAPES.git
%cd SCAPES

# Install dependencies
!pip install -r requirements.txt

# Ensure Python can find the SCAPES package
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("SCAPES setup complete.")

## Mount your Google Drive (optional)

If running in Colab, you may want to mount your Google Drive to transfer files. This is optional — you can also simply upload audio files by dragging and dropping them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **1. Dataset Preparation**

In this section, we will extract EnCodec audio **atoms** (sequences of continuous embeddings) and precompute **semantic** embeddings (CLAP).


## 1.1. Dataset Structure

First, make sure your data follows the structure below. Create a new directory for your dataset with two subdirectories: `raw/` (for your WAV files) and `config/` (for the configuration files). Copy the gin and CSV templates from `SCAPES/config/` into your `config/` directory to get started. In this notebook we will rely mostly on default parameters, and in particular dataprep.gin will not be changed.

```
<dataset>/
├── raw/
│   ├── audio_file_1.wav
│   ├── ...
│   └── audio_file_N.wav
└── config/
    ├── dataprep.gin
    ├── training.gin
    └── cherry_picking.csv
```

## 1.2. Define the path for your dataset run all dataprep functions

In [ ]:
from quickstart.wrappers import data_preparation

DATASET_PATH = "/home/esteban/Documents/projects/dev_SCAPES/datasets/micro_microtex" # <--------------------------- Change this to the path of your dataset

data_preparation(DATASET_PATH)

## **Restart the kernel and run the following cell before training**

In [ ]:
%cd SCAPES
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)
print("SCAPES ready.")

# **2. Training**

## 2.1. Build semantics folder from cherry-picking CSV (optional: in case you want to use the online GUI)

The `config/cherry_picking.csv` defines audio segments from which semantic cards can be extracted. These cards can then be used in the online GUI. If you are not interested in running the model online, you can safely delete this file.

Fill in the CSV following the template below, pointing to the filename and the time segment from which the semantics should be extracted:

| filename | start_sec | end_sec | flag | icon | description |
|---|---|---|---|---|---|
| secret_mars_recordings.wav | 0 | 5 | alien | 🛸 | alien sounds recorded on Mars |
|...|...|...|...|...|...|


## 2.2. Set the size of your model and train

Go into the file training.gin and dcide the size of your model. Pick from "small", "medium" and "large". The larger, the more it takes to train. As a benchmark, using a T4 (Colab free GPU) and a dataset of 20 minutes, will take 1 hour to train the small model and 4 hours to train the large model. Once you have edited the file accordingly, continue with the following cell.

In [ ]:
DATASET_PATH = "path/to/your/dataset" # <--------------------------- Change this to the path of your dataset
MODEL_PATH   = "/content/my_model" # <------------------------------ Change this to the path where you want to save the model

from SCAPES.quickstart.wrappers import train

train(DATASET_PATH, MODEL_PATH)

## **Restart the kernel and run the following cell before training**

In [ ]:
%cd SCAPES
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)
print("SCAPES ready.")

# **3. Inference**

## 3.1. Online Inference

In order to do online inference, you will need to upload your model to a Hugging Face model repository. This can be done directly in the browser using a free account.

First, clean up your model checkpoints and keep only the one you prefer. If in doubt, use the ones called `best`. You will also need to create a markdown file called `info.md` — write whatever you want in it, as this information will appear as the model description in the GUI.

```
<model>/
├── info.md
├── checkpoints/
│   ├── best_flow_model.pt
│   ├── best_local_encoder.pt
│   └── inference.gin
└── semantics/
    ├── mars_sounds.pt
    ├── ...
    └── semantics.csv
```

Once the structure is correct, create a Hugging Face account, look for the option to create a new model repository, and follow the instructions. This will generate a git repository. Then use the browser File Uploader to drag and drop your full model folder into it, fill in the commit message at the bottom of the page, and press Commit.

Once the upload is done, copy the link to your repository and paste it into the Add Models option in our online GUI available at: https://huggingface.co/spaces/cordutie/SCAPES-demo

Have fun!

## 3.2. Local Inference

Load the trained model and generate audio in various modes.
All model and architecture settings come from `<model_dir>/checkpoints/inference.gin`
(generated during training).

**Key parameters:**
- `NFE`: Number of Function Evaluations for the ODE solver (higher = better quality, default: 32)
- `cfg_scale`: Classifier-free guidance scale (higher = more "typical" generations, default: 3.0)
- `TF` (Teacher Forcing):
  - `True` — use full ground-truth context (reconstruction)
  - `"partial"` — use 0.5 s of audio, then generate freely (semantic reconstruction)
  - `False` — fully generative (no teacher forcing)
- `decode_method`: `"ola_smooth"` (overlap-add with smoothing, default) or `"ola_linear"`

In [ ]:
# === CHANGE THIS ===
MODEL_PATH = "path/to/your/model"

import torch
from pathlib import Path
from IPython.display import Audio, display

from SCAPES.inference.FlowInference import (
    FlowInference,
    run_resynthesis_pipeline,
    run_batch_resynthesis_pipeline,
    run_interpolation_pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# checkpoint: 'best' (best validation), 'last' (final), or an epoch number
engine = FlowInference(
    model_dir=MODEL_PATH,
    device=device,
    verbose=True,
    checkpoint="best",
)
print(f"Loaded model from {MODEL_PATH}")

### 3.2.1. Full reconstruction (teacher-forced)

Reconstructs audio from its full latent representation. The model gets ground-truth
atoms as context — measures autoencoder fidelity.

In [ ]:
audio_path = "path/to/your/audio.wav"
duration = 10

audio_tensor = engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000]
print("Original:", Path(audio_path).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(
    engine=engine,
    audio_path=audio_path,
    duration=duration,
    play=True,
    TF=True,
    NFE=16,  # fewer NFE is fine for TF reconstruction
)

### 3.2.2. Semantic reconstruction (partial TF)

Uses only the first 0.5 s to extract the CLAP embedding, then generates the rest
from scratch. Same semantic content, different acoustic realization.

In [ ]:
audio_path = "path/to/your/audio.wav"
duration = 10

audio_tensor = engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000]
print("Original:", Path(audio_path).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(
    engine=engine,
    audio_path=audio_path,
    duration=duration,
    play=True,
    TF="partial",
    NFE=32,
)

### 3.2.3. Semantic interpolation

Interpolates between two audio files' semantic embeddings.
`stickyness` controls transition sharpness (1.0 = linear, higher = sharper midpoint).

In [ ]:
audio_1 = "path/to/start.wav"
audio_2 = "path/to/end.wav"

for path, label in [(audio_1, "Start"), (audio_2, "End")]:
    t = engine.load_audio_to_tensor(path)[:, :, :5 * 48000]
    print(f"{label}: {Path(path).stem}")
    display(Audio(t.detach().cpu().numpy()[0], rate=48000))

output = run_interpolation_pipeline(
    engine=engine,
    audio_path_1=audio_1,
    audio_path_2=audio_2,
    timeline_size=200,
    stay_time=20,
    stickyness=3.0,
    play=True,
    NFE=32,
    context_static=False,
)